# Voyexa AI — Model 5: Combined Ranking Blend
The actual scoring function the backend calls after the **rule layer** has already filtered candidates by budget / difficulty-safety / season / state.

```
final_score = w_similarity * similarity_to_user_profile
            + w_popularity * predicted_popularity   (Model 2, normalized)
            + w_hidden_gem * hidden_gem_probability (Model 1)
```

Weights are tunable — defaults favor matching the user's stated interests most, popularity next, with a hidden-gem "discovery" nudge. Requires notebooks 03, 04 and 05 to have been run first (their artifacts are loaded here).

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

DATA_PATH = os.path.join("..", "..", "data", "processed", "places_processed.csv")
ARTIFACTS_DIR = os.path.join("..", "artifacts")

df = pd.read_csv(DATA_PATH)

hidden_gem_clf = joblib.load(os.path.join(ARTIFACTS_DIR, "hidden_gem_classifier.pkl"))
hidden_gem_features = joblib.load(os.path.join(ARTIFACTS_DIR, "hidden_gem_features.pkl"))

popularity_reg = joblib.load(os.path.join(ARTIFACTS_DIR, "popularity_regressor.pkl"))
popularity_features = joblib.load(os.path.join(ARTIFACTS_DIR, "popularity_features.pkl"))

sim_feature_matrix = joblib.load(os.path.join(ARTIFACTS_DIR, "similarity_feature_matrix.pkl"))
vec_cols = joblib.load(os.path.join(ARTIFACTS_DIR, "similarity_vec_cols.pkl"))

print("All model artifacts loaded.")

All model artifacts loaded.


## Rebuild feature frames + get predictions for every place

In [2]:
zone_dummies = pd.get_dummies(df["Zone"], prefix="zone")
terrain_dummies = pd.get_dummies(df["Terrain_Type"], prefix="terrain")

def build_X(feature_list):
    base = pd.concat([df, zone_dummies, terrain_dummies], axis=1)
    for col in feature_list:
        if col not in base.columns:
            base[col] = 0
    return base[feature_list]

X_hidden_gem = build_X(hidden_gem_features)
X_popularity = build_X(popularity_features)
df = df.copy()

df["pred_hidden_gem_prob"] = hidden_gem_clf.predict_proba(X_hidden_gem)[:, 1]
pred_pop_raw = popularity_reg.predict(X_popularity)
df["pred_popularity_norm"] = MinMaxScaler().fit_transform(pred_pop_raw.reshape(-1, 1))

df[["Place_Name", "pred_hidden_gem_prob", "pred_popularity_norm"]].head()

,Place_Name,pred_hidden_gem_prob,pred_popularity_norm
0,Dal Lake,0.225607,0.605579
1,Shalimar Bagh,0.760668,0.744199
2,Sonamarg,0.792306,1.000000
3,Gulmarg,0.797987,0.968478
4,Noori Chamb Waterfall,0.197512,0.124187


## The blended recommend() function

In [3]:
def recommend(interest_tags, suitable_for, candidate_place_ids=None, top_n=10,
              w_similarity=0.5, w_popularity=0.3, w_hidden_gem=0.2):
    query = pd.Series(0.0, index=vec_cols)
    for tag in interest_tags:
        col = f"interest_{tag.lower().replace(' ', '_')}"
        if col in query.index:
            query[col] = 1.0
    for s in suitable_for:
        col = f"suitable_{s.lower().replace(' ', '_')}"
        if col in query.index:
            query[col] = 1.0
    query_full = pd.concat([query, pd.Series({"difficulty_norm": 0.0, "cost_norm": 0.0})])

    candidates = df if candidate_place_ids is None else df[df["place_id"].isin(candidate_place_ids)]
    cand_idx = candidates.index
    cand_matrix = sim_feature_matrix.loc[cand_idx]

    sims = cosine_similarity([query_full.values], cand_matrix.values)[0]

    final_score = (
        w_similarity * sims
        + w_popularity * candidates["pred_popularity_norm"].values
        + w_hidden_gem * candidates["pred_hidden_gem_prob"].values
    )

    result = candidates.copy()
    result["similarity"] = sims.round(3)
    result["final_score"] = final_score.round(3)
    result = result.sort_values("final_score", ascending=False).head(top_n)

    return result[["place_id", "Place_Name", "City", "State", "Interest_Tags",
                    "Popularity_Rating", "Is_Hidden_Gem", "similarity",
                    "pred_popularity_norm", "pred_hidden_gem_prob", "final_score"]]

## Demo: default weights

In [4]:
recommend(interest_tags=["Adventure", "Trekking"], suitable_for=["Friends"], top_n=8)

,place_id,Place_Name,City,State,Interest_Tags,Popularity_Rating,Is_Hidden_Gem,similarity,pred_popularity_norm,pred_hidden_gem_prob,final_score
1011,1014,Kalsubai Peak,Ahmednagar,Maharashtra,"Trekking,Adventure",4.9,True,0.856,0.906054,0.937171,0.887
1138,1141,Sandakphu,Darjeeling,West Bengal,"Trekking,Adventure",4.8,True,0.856,0.827822,0.948567,0.866
986,989,Harishchandragad,Ahmednagar,Maharashtra,"Trekking,Adventure",4.8,True,0.863,0.832930,0.909532,0.864
987,990,Kalavantin Durg,Panvel,Maharashtra,"Trekking,Adventure",4.7,True,0.863,0.832930,0.909532,0.864
824,827,Roopkund Lake,Chamoli,Uttarakhand,"Trekking,Adventure",4.7,True,0.863,0.783078,0.973355,0.861
811,814,Kafni Glacier,Bageshwar,Uttarakhand,"Trekking,Adventure",4.6,True,0.856,0.774824,0.981192,0.857
810,813,Pindari Glacier,Bageshwar,Uttarakhand,"Trekking,Adventure",4.7,True,0.856,0.774824,0.984901,0.857
809,812,Milam Glacier,Pithoragarh,Uttarakhand,"Trekking,Adventure",4.7,True,0.856,0.774824,0.960367,0.853


## Demo: favor hidden gems more (discovery mode)

In [5]:
recommend(
    interest_tags=["Adventure", "Trekking"], suitable_for=["Friends"], top_n=8,
    w_similarity=0.3, w_popularity=0.2, w_hidden_gem=0.5,
)

,place_id,Place_Name,City,State,Interest_Tags,Popularity_Rating,Is_Hidden_Gem,similarity,pred_popularity_norm,pred_hidden_gem_prob,final_score
1011,1014,Kalsubai Peak,Ahmednagar,Maharashtra,"Trekking,Adventure",4.9,True,0.856,0.906054,0.937171,0.907
810,813,Pindari Glacier,Bageshwar,Uttarakhand,"Trekking,Adventure",4.7,True,0.856,0.774824,0.984901,0.904
811,814,Kafni Glacier,Bageshwar,Uttarakhand,"Trekking,Adventure",4.6,True,0.856,0.774824,0.981192,0.902
824,827,Roopkund Lake,Chamoli,Uttarakhand,"Trekking,Adventure",4.7,True,0.863,0.783078,0.973355,0.902
1138,1141,Sandakphu,Darjeeling,West Bengal,"Trekking,Adventure",4.8,True,0.856,0.827822,0.948567,0.897
809,812,Milam Glacier,Pithoragarh,Uttarakhand,"Trekking,Adventure",4.7,True,0.856,0.774824,0.960367,0.892
986,989,Harishchandragad,Ahmednagar,Maharashtra,"Trekking,Adventure",4.8,True,0.863,0.832930,0.909532,0.880
987,990,Kalavantin Durg,Panvel,Maharashtra,"Trekking,Adventure",4.7,True,0.863,0.832930,0.909532,0.880
